In [35]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\Fiorella\OneDrive\VSCODE PROJECTS\Projects\DTSC 2\Personal Project Part 1")
RAW_DIR = PROJECT_DIR / "data" / "raw_data"
PROCESSED_DIR = PROJECT_DIR / "notebooks" / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "notebooks" / "output"

# Load raw files.
gdf_canopy_raw = gpd.read_file(
    RAW_DIR / "tree_canopy" / "charlotte_canopy.geojson",
    engine="pyogrio",
)
df_census_raw = pd.read_csv(RAW_DIR / "census" / "mecklenburg_census.csv")
gdf_heat_raw = gpd.read_file(
    RAW_DIR / "heat_island" / "charlotte-north-carolina_af_trav.shp",
    engine="pyogrio",
)

print("Canopy raw shape:", gdf_canopy_raw.shape)
print("Census raw shape:", df_census_raw.shape)
print("Heat raw shape:", gdf_heat_raw.shape)

Canopy raw shape: (1135, 46)
Census raw shape: (305, 7)
Heat raw shape: (23558, 4)


In [36]:
# The canopy dataset contains multiple geography types (City, Council District, NPA, Block Group, etc.)
# We only want block groups — the finest geography with full canopy data
gdf_canopy = gdf_canopy_raw[gdf_canopy_raw['Geography'] == 'Block Group'].copy()
gdf_canopy = gdf_canopy.reset_index(drop=True)

print('Block groups retained:', len(gdf_canopy))
print('Missing values in key columns:')
key_cols = [
    'Geography_Value', 'UrbanTreeCanopyAreaPercent', 'TreeCanopyArea2022Percent',
    'UrbanTreeCanopyAreaPer18', 'NetTreeCanopyChange18_22',
    'RelativeTreeCanopyChange18_22', 'ImperviousAreaPercent'
]
print(gdf_canopy[key_cols].isna().sum())

Block groups retained: 624
Missing values in key columns:
Geography_Value                  0
UrbanTreeCanopyAreaPercent       0
TreeCanopyArea2022Percent        0
UrbanTreeCanopyAreaPer18         0
NetTreeCanopyChange18_22         0
RelativeTreeCanopyChange18_22    0
ImperviousAreaPercent            0
dtype: int64


In [37]:
# Rename columns to readable names and keep only what's needed for analysis
gdf_canopy = gdf_canopy.rename(columns={
    'Geography_Value': 'GEOID_bg',
    'UrbanTreeCanopyAreaPercent': 'canopy_pct_2022',
    'TreeCanopyArea2022Percent': 'canopy_pct_2022_alt',
    'UrbanTreeCanopyAreaPer18': 'canopy_pct_2018',
    'NetTreeCanopyChange18_22': 'canopy_change_acres',
    'RelativeTreeCanopyChange18_22': 'canopy_change_pct',
    'ImperviousAreaPercent': 'impervious_pct',
    'PossiblePlantingVegetationPer': 'planting_opportunity_pct',
})

keep_cols = [
    'GEOID_bg', 'canopy_pct_2022', 'canopy_pct_2018',
    'canopy_change_pct', 'canopy_change_acres',
    'impervious_pct', 'planting_opportunity_pct', 'geometry'
]
gdf_canopy = gdf_canopy[keep_cols]

# Extract tract GEOID (first 11 digits of 12-digit block group GEOID)
# Block group GEOID: state(2) + county(3) + tract(6) + bg(1) = 12 digits
# Tract GEOID: state(2) + county(3) + tract(6) = 11 digits
gdf_canopy['GEOID_tract'] = gdf_canopy['GEOID_bg'].str[:11]

print('Canopy columns kept:', list(gdf_canopy.columns))
print('Canopy CRS:', gdf_canopy.crs)
gdf_canopy.head()

Canopy columns kept: ['GEOID_bg', 'canopy_pct_2022', 'canopy_pct_2018', 'canopy_change_pct', 'canopy_change_acres', 'impervious_pct', 'planting_opportunity_pct', 'geometry', 'GEOID_tract']
Canopy CRS: EPSG:4326


,GEOID_bg,canopy_pct_2022,canopy_pct_2018,canopy_change_pct,canopy_change_acres,impervious_pct,planting_opportunity_pct,geometry,GEOID_tract
0,371190015082,66.45,63.858131,0.670960,2.587169,30.62,17.44,"POLYGON ((-80.71656 35.24086, -80.71642 35.239...",37119001508
1,371190054062,53.17,52.473882,0.205861,0.695350,19.89,24.71,"POLYGON ((-80.85015 35.28699, -80.8494 35.2853...",37119005406
2,371190054061,42.62,44.842770,-0.407754,-2.219978,32.10,24.04,"POLYGON ((-80.85826 35.30497, -80.85823 35.304...",37119005406
3,371190055322,56.03,55.345393,0.178887,0.680986,17.58,20.97,"POLYGON ((-80.72109 35.33349, -80.72136 35.333...",37119005532
4,371190058642,42.96,32.852438,7.419583,10.103618,35.41,20.89,"POLYGON ((-80.81124 35.02156, -80.81126 35.021...",37119005864


In [38]:
min_canopy = gdf_canopy['canopy_pct_2022'].min()
max_canopy = gdf_canopy['canopy_pct_2022'].max()
print(f"Min Canopy: {min_canopy:.1f}%, Max Canopy: {max_canopy:.1f}%")

Min Canopy: 3.1%, Max Canopy: 82.5%


In [39]:
# GEOID was stored as integer — must convert to zero-padded string before any merge
# 11-digit tract GEOID: state(2) + county(3) + tract(6)
df_census = df_census_raw.copy()

df_census['GEOID_tract'] = df_census['GEOID'].astype(str).str.zfill(11)

# median_income is stored as int — some rows may be coded as -666666666 (Census suppression code)
df_census['median_income'] = pd.to_numeric(df_census['median_income'], errors='coerce')
df_census['median_income'] = df_census['median_income'].replace(-666666666, np.nan)

print('Census GEOID sample:', df_census['GEOID_tract'].iloc[0])
print('Suppressed income values:', df_census['median_income'].isna().sum())
print('Census shape:', df_census.shape)

keep_census = ['GEOID_tract', 'total_population', 'median_income']
df_census = df_census[keep_census]
df_census.head()

Census GEOID sample: 37119000101
Suppressed income values: 5
Census shape: (305, 8)


,GEOID_tract,total_population,median_income
0,37119000101,1148,101587.0
1,37119000102,2741,123650.0
2,37119000103,2042,131398.0
3,37119000104,1619,109896.0
4,37119000301,954,82500.0


In [40]:
# Left join: keep all block groups, bring in income from parent tract
# Every block group within the same tract gets the same income value
gdf_merged = gdf_canopy.merge(df_census, on='GEOID_tract', how='left')

match_rate = gdf_merged['median_income'].notna().mean()
print(f'Income match rate: {match_rate:.1%}')
print(f'Rows after merge: {len(gdf_merged)}')
print(f'Block groups with no income data: {gdf_merged["median_income"].isna().sum()}')

gdf_merged.head()

Income match rate: 98.9%
Rows after merge: 624
Block groups with no income data: 7


,GEOID_bg,canopy_pct_2022,canopy_pct_2018,canopy_change_pct,canopy_change_acres,impervious_pct,planting_opportunity_pct,geometry,GEOID_tract,total_population,median_income
0,371190015082,66.45,63.858131,0.670960,2.587169,30.62,17.44,"POLYGON ((-80.71656 35.24086, -80.71642 35.239...",37119001508,6227,67808.0
1,371190054062,53.17,52.473882,0.205861,0.695350,19.89,24.71,"POLYGON ((-80.85015 35.28699, -80.8494 35.2853...",37119005406,3967,48337.0
2,371190054061,42.62,44.842770,-0.407754,-2.219978,32.10,24.04,"POLYGON ((-80.85826 35.30497, -80.85823 35.304...",37119005406,3967,48337.0
3,371190055322,56.03,55.345393,0.178887,0.680986,17.58,20.97,"POLYGON ((-80.72109 35.33349, -80.72136 35.333...",37119005532,4137,67108.0
4,371190058642,42.96,32.852438,7.419583,10.103618,35.41,20.89,"POLYGON ((-80.81124 35.02156, -80.81126 35.021...",37119005864,4922,153977.0


In [41]:
gdf_merged.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 624 entries, 0 to 623
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   GEOID_bg                  624 non-null    str     
 1   canopy_pct_2022           624 non-null    float64 
 2   canopy_pct_2018           624 non-null    float64 
 3   canopy_change_pct         624 non-null    float64 
 4   canopy_change_acres       624 non-null    float64 
 5   impervious_pct            624 non-null    float64 
 6   planting_opportunity_pct  624 non-null    float64 
 7   geometry                  624 non-null    geometry
 8   GEOID_tract               624 non-null    str     
 9   total_population          624 non-null    int64   
 10  median_income             617 non-null    float64 
dtypes: float64(7), geometry(1), int64(1), str(2)
memory usage: 53.8 KB


In [42]:
# Heat data is 23,558 GPS points from CHARP's afternoon traverse (one day, July 2024)
# Strategy: spatial join each heat point to the block group it falls inside,
# then compute mean heat index per block group

gdf_heat = gdf_heat_raw.copy()

# Confirm both are in EPSG:4326 before spatial join
print('Heat CRS:', gdf_heat.crs)
print('Canopy CRS:', gdf_merged.crs)

# Spatial join: tag each heat point with the block group it falls inside
heat_in_bg = gpd.sjoin(
    gdf_heat,
    gdf_merged[['GEOID_bg', 'geometry']],
    how='inner',
    predicate='within'
)

print(f'Heat points matched to a block group: {len(heat_in_bg)} of {len(gdf_heat)}')
print(f'Block groups that received at least one heat point: {heat_in_bg["GEOID_bg"].nunique()}')

Heat CRS: EPSG:4326
Canopy CRS: EPSG:4326
Heat points matched to a block group: 23558 of 23558
Block groups that received at least one heat point: 196


In [43]:
gdf_heat.info()
gdf_heat.head()
gdf_heat[['t_f', 'rh', 'hi_f']].describe()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 23558 entries, 0 to 23557
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   t_f       23558 non-null  float64 
 1   rh        23558 non-null  float64 
 2   hi_f      23558 non-null  float64 
 3   geometry  23558 non-null  geometry
dtypes: float64(3), geometry(1)
memory usage: 736.3 KB


,t_f,rh,hi_f
count,23558.000000,23558.000000,23558.000000
mean,94.958511,36.760867,97.181565
std,2.495399,6.299144,2.062873
min,86.300000,23.600000,89.800000
25%,93.600000,32.200000,95.800000
50%,95.300000,34.900000,97.000000
75%,96.700000,40.700000,98.500000
max,105.000000,59.100000,106.600000


In [44]:
# Aggregate mean heat index per block group
heat_by_bg = heat_in_bg.groupby('GEOID_bg').agg(
    mean_heat_index=('hi_f', 'mean'),
    mean_temp_f=('t_f', 'mean'),
    heat_point_count=('hi_f', 'count')
).reset_index()

print('Block groups with heat data:', len(heat_by_bg))
print('Mean heat index range:', heat_by_bg['mean_heat_index'].min(), 'to', heat_by_bg['mean_heat_index'].max())
heat_by_bg.head()
heat_by_bg.info()
heat_by_bg.describe()

Block groups with heat data: 196
Mean heat index range: 92.05057471264368 to 102.57352941176471
<class 'pandas.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GEOID_bg          196 non-null    str    
 1   mean_heat_index   196 non-null    float64
 2   mean_temp_f       196 non-null    float64
 3   heat_point_count  196 non-null    int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 6.3 KB


,mean_heat_index,mean_temp_f,heat_point_count
count,196.000000,196.000000,196.000000
mean,97.309669,94.991936,120.193878
std,1.831282,2.384854,97.921335
min,92.050575,88.480000,1.000000
25%,96.260918,93.722946,50.000000
50%,97.172462,95.259959,107.000000
75%,98.567225,96.622338,153.000000
max,102.573529,100.832353,578.000000


In [45]:
# Join heat summaries to the full canopy and census analysis table.
gdf_analysis = gdf_merged.merge(heat_by_bg, on="GEOID_bg", how="left")

heat_coverage = gdf_analysis["mean_heat_index"].notna().mean()
print(
    f"Block groups with heat data: {gdf_analysis['mean_heat_index'].notna().sum()} "
    f"({heat_coverage:.1%})"
)
print(
    f"Block groups without heat data: "
    f"{gdf_analysis['mean_heat_index'].isna().sum()}"
)
print(f"Analysis dataset shape: {gdf_analysis.shape}")

gdf_analysis.head()

Block groups with heat data: 196 (31.4%)
Block groups without heat data: 428
Analysis dataset shape: (624, 14)


,GEOID_bg,canopy_pct_2022,canopy_pct_2018,canopy_change_pct,canopy_change_acres,impervious_pct,planting_opportunity_pct,geometry,GEOID_tract,total_population,median_income,mean_heat_index,mean_temp_f,heat_point_count
0,371190015082,66.45,63.858131,0.670960,2.587169,30.62,17.44,"POLYGON ((-80.71656 35.24086, -80.71642 35.239...",37119001508,6227,67808.0,NaN,NaN,NaN
1,371190054062,53.17,52.473882,0.205861,0.695350,19.89,24.71,"POLYGON ((-80.85015 35.28699, -80.8494 35.2853...",37119005406,3967,48337.0,NaN,NaN,NaN
2,371190054061,42.62,44.842770,-0.407754,-2.219978,32.10,24.04,"POLYGON ((-80.85826 35.30497, -80.85823 35.304...",37119005406,3967,48337.0,96.022881,95.55791,354.0
3,371190055322,56.03,55.345393,0.178887,0.680986,17.58,20.97,"POLYGON ((-80.72109 35.33349, -80.72136 35.333...",37119005532,4137,67108.0,NaN,NaN,NaN
4,371190058642,42.96,32.852438,7.419583,10.103618,35.41,20.89,"POLYGON ((-80.81124 35.02156, -80.81126 35.021...",37119005864,4922,153977.0,NaN,NaN,NaN


In [46]:
# Check geometry validity before exporting the analysis dataset.
invalid = (~gdf_analysis.geometry.is_valid).sum()
print(f"Invalid geometries: {invalid}")
if invalid > 0:
    gdf_analysis.geometry = gdf_analysis.geometry.buffer(0)
    print("Fixed invalid geometries with buffer(0)")

print("\nAnalysis columns:")
print(gdf_analysis.dtypes)

print("\nMissing values per column:")
print(gdf_analysis.isna().sum())

Invalid geometries: 0

Analysis columns:
GEOID_bg                         str
canopy_pct_2022              float64
canopy_pct_2018              float64
canopy_change_pct            float64
canopy_change_acres          float64
impervious_pct               float64
planting_opportunity_pct     float64
geometry                    geometry
GEOID_tract                      str
total_population               int64
median_income                float64
mean_heat_index              float64
mean_temp_f                  float64
heat_point_count             float64
dtype: object

Missing values per column:
GEOID_bg                      0
canopy_pct_2022               0
canopy_pct_2018               0
canopy_change_pct             0
canopy_change_acres           0
impervious_pct                0
planting_opportunity_pct      0
geometry                      0
GEOID_tract                   0
total_population              0
median_income                 7
mean_heat_index             428
mean_temp_f    

In [47]:
# Export paths are defined once in the first cell.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Neighborhood enrichment is added in the next cell before export.")

Neighborhood enrichment is added in the next cell before export.


In [48]:
gdf_heat_subset.info()
gdf_heat_subset.head()
gdf_heat_subset[['mean_heat_index', 'mean_temp_f', 'heat_point_count']].describe()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 196 entries, 2 to 616
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   GEOID_bg                  196 non-null    str     
 1   canopy_pct_2022           196 non-null    float64 
 2   canopy_pct_2018           196 non-null    float64 
 3   canopy_change_pct         196 non-null    float64 
 4   canopy_change_acres       196 non-null    float64 
 5   impervious_pct            196 non-null    float64 
 6   planting_opportunity_pct  196 non-null    float64 
 7   geometry                  196 non-null    geometry
 8   GEOID_tract               196 non-null    str     
 9   total_population          196 non-null    int64   
 10  median_income             196 non-null    float64 
 11  mean_heat_index           196 non-null    float64 
 12  mean_temp_f               196 non-null    float64 
 13  heat_point_count          196 non-null    float

,mean_heat_index,mean_temp_f,heat_point_count
count,196.000000,196.000000,196.000000
mean,97.309669,94.991936,120.193878
std,1.831282,2.384854,97.921335
min,92.050575,88.480000,1.000000
25%,96.260918,93.722946,50.000000
50%,97.172462,95.259959,107.000000
75%,98.567225,96.622338,153.000000
max,102.573529,100.832353,578.000000


In [51]:
# Add verified NPA-to-neighborhood labels for map and neighborhood-level analysis.
gdf_npa = gdf_canopy_raw[gdf_canopy_raw["Geography"] == "NPA"].copy()
gdf_npa["NPA_ID"] = gdf_npa["Geography_Value"].astype(str).str.strip()

neighborhood_crosswalk = {
    "15": "Uptown",
    "16": "First Ward",
    "17": "Second Ward",
    "18": "Third Ward",
    "19": "Fourth Ward",
    "80": "South End",
    "82": "Brookhill",
    "86": "Enderly Park",
    "88": "Hoskins",
    "94": "Hidden Valley",
    "96": "Druid Hills",
    "123": "Plaza Midwood",
    "127": "Windsor Park",
    "139": "NoDa",
    "167": "Eastland",
    "246": "University City",
    "325": "Myers Park",
    "326": "Eastover",
    "328": "Foxcroft",
}
gdf_npa["neighborhood_name"] = gdf_npa["NPA_ID"].map(neighborhood_crosswalk)

# Use block-group centroids for a deterministic NPA assignment.
gdf_analysis_proj = gdf_analysis.to_crs(epsg=2264)
gdf_npa_proj = gdf_npa.to_crs(epsg=2264)
centroids = gdf_analysis_proj[["GEOID_bg", "geometry"]].copy()
centroids["geometry"] = centroids.geometry.centroid

tagged = gpd.sjoin(
    centroids,
    gdf_npa_proj[["NPA_ID", "neighborhood_name", "geometry"]],
    how="left",
    predicate="within",
).drop_duplicates(subset=["GEOID_bg"])

# Make this cell safe to rerun after a previous enrichment.
gdf_analysis = gdf_analysis.drop(
    columns=["NPA_ID", "neighborhood_name"],
    errors="ignore",
)
gdf_analysis = gdf_analysis.merge(
    tagged[["GEOID_bg", "NPA_ID", "neighborhood_name"]],
    on="GEOID_bg",
    how="left",
)

# Export only after all enrichment is complete.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
gdf_analysis.to_file(
    PROCESSED_DIR / "charlotte_analysis.geojson",
    driver="GeoJSON",
)
gdf_heat_subset = gdf_analysis[
    gdf_analysis["mean_heat_index"].notna()
].copy()
gdf_heat_subset.to_file(
    PROCESSED_DIR / "charlotte_analysis_heat_subset.geojson",
    driver="GeoJSON",
)

print(f"Analysis dataset exported: {gdf_analysis.shape}")
print(
    f"Named neighborhood assignments: "
    f"{gdf_analysis['neighborhood_name'].notna().sum()}"
)

Analysis dataset exported: (624, 20)
Named neighborhood assignments: 27
